# Data Visualization

## Loading Libraries

In [148]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [149]:
OUTPUT_DIR = "../Data/citibike/"

## Monthly Data

In [150]:
weather_daily = pd.read_csv('../data/citibike/JC/jersey_weather_2025.csv')
citibike_df = pd.read_csv('../data/citibike/JC/JC2025_Enriched.csv')

In [151]:
weather_daily['date'] = pd.to_datetime(weather_daily['date'], errors="coerce")
citibike_df['date'] = pd.to_datetime(citibike_df['date'], errors="coerce")

In [152]:
citibike_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 998281 entries, 0 to 998280
Data columns (total 22 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   ride_id             998281 non-null  str           
 1   rideable_type       998281 non-null  str           
 2   started_at          998281 non-null  str           
 3   ended_at            998281 non-null  str           
 4   start_station_name  998281 non-null  str           
 5   start_station_id    998281 non-null  str           
 6   end_station_name    998281 non-null  str           
 7   end_station_id      998281 non-null  str           
 8   start_lat           998281 non-null  float64       
 9   start_lng           998281 non-null  float64       
 10  end_lat             998281 non-null  float64       
 11  end_lng             998281 non-null  float64       
 12  member_casual       998281 non-null  str           
 13  ride_duration_min   998281 non-null  flo

In [153]:
monthly_rides = (citibike_df
                 .groupby('month',as_index=False)
                 .agg(number_of_rides = ('ride_id',"count"))
                 )
monthly_rides

,month,number_of_rides
0,2024-12,2
1,2025-01,50477
2,2025-02,45131
3,2025-03,73124
4,2025-04,81295
5,2025-05,92880
6,2025-06,96736
7,2025-07,107374
8,2025-08,108001
9,2025-09,115580


In [154]:
fig = px.bar(
    monthly_rides,
    x="month",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Month"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Rides",
)

fig.show()

## Number of Rides by Season

In [155]:
season_rides = (
    citibike_df
    .groupby("season", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]

season_rides["season"] = pd.Categorical(
    season_rides["season"],
    categories=season_order,
    ordered=True
)

season_rides = season_rides.sort_values("season")

season_rides

,season,number_of_rides
3,Winter,143472
1,Spring,247299
2,Summer,312111
0,Autumn,295399


In [156]:
fig = px.bar(
    season_rides,
    x="season",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Season",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Number of Rides"
)

fig.show()

### Merge with Weather Data

In [157]:
daily_rides = (
    citibike_df
    .groupby("date", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)
daily_rides["date"] = pd.to_datetime(daily_rides["date"])
daily_rides.head()

,date,number_of_rides
0,2024-12-31,2
1,2025-01-01,1174
2,2025-01-02,1709
3,2025-01-03,1764
4,2025-01-04,1336


In [158]:
weather_daily.head()

,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2025-01-01,10.9,3.9,7.4,4.5,4.5,0.0,23.2
1,2025-01-02,5.4,0.3,2.6,0.0,0.0,0.0,25.1
2,2025-01-03,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
3,2025-01-04,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1
4,2025-01-05,0.3,-3.6,-2.2,0.0,0.0,0.0,19.9


In [159]:
bike_weather_daily = daily_rides.merge(
    weather_daily,
    on="date",
    how="left"
)

bike_weather_daily.head()

,date,number_of_rides,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2024-12-31,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01,1174,10.9,3.9,7.4,4.5,4.5,0.0,23.2
2,2025-01-02,1709,5.4,0.3,2.6,0.0,0.0,0.0,25.1
3,2025-01-03,1764,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
4,2025-01-04,1336,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1


In [160]:
bike_weather_daily.isna().sum()

date                   0
number_of_rides        0
temperature_2m_max     1
temperature_2m_min     1
temperature_2m_mean    1
precipitation_sum      1
rain_sum               1
snowfall_sum           1
wind_speed_10m_max     1
dtype: int64

### Rides vs Average Temperature

In [161]:
fig = px.scatter(
    bike_weather_daily,
    x="temperature_2m_mean",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Average Temperature"
)

fig.update_layout(
    xaxis_title="Average Daily Temperature",
    yaxis_title="Number of Rides"
)

fig.show()

### Rides vs Wind Speed

In [162]:
fig = px.scatter(
    bike_weather_daily,
    x="wind_speed_10m_max",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Maximum Wind Speed"
)

fig.update_layout(
    xaxis_title="Maximum Wind Speed",
    yaxis_title="Number of Rides"
)

fig.show()

### Rides vs Precipitation

In [163]:
fig = px.scatter(
    bike_weather_daily,
    x="precipitation_sum",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Precipitation"
)

fig.update_layout(
    xaxis_title="Daily Precipitation",
    yaxis_title="Number of Rides"
)

fig.show()

### Rides vs Temperature

In [164]:
import plotly.graph_objects as go


fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["number_of_rides"],
        mode="lines",
        name="Daily Rides",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["temperature_2m_mean"],
        mode="lines",
        name="Daily Average Temperature",
        yaxis="y2"
    )
)

fig.update_layout(
    title="Daily Rides and Daily Average Temperature",
    xaxis=dict(
        title="Day"
    ),
    yaxis=dict(
        title="Daily Rides",
        side="left"
    ),
    yaxis2=dict(
        title="Daily Average Temperature",
        overlaying="y",
        side="right"
    ),
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()

### Number of Rides by DOW

In [165]:
days = (citibike_df
.groupby("day_of_week", as_index=False)
.agg(number_of_rides=("ride_id", "count"))
)

In [166]:
day_order = [
"Monday",
"Tuesday",
"Wednesday",
"Thursday",
"Friday",
"Saturday",
"Sunday"
]

In [167]:
days["day_of_week"] = pd.Categorical(
days["day_of_week"],
categories=day_order,
ordered=True
)

days = days.sort_values("day_of_week")

In [168]:
days["Type"] = "Normal"

days.loc[
days["number_of_rides"].idxmax(),
"Type"
] = "Highest"

days.loc[
days["number_of_rides"].idxmin(),
"Type"
] = "Lowest"

In [169]:
fig = px.bar(
days,
x="day_of_week",
y="number_of_rides",
color="Type",
text_auto=True,
title="Number of Citi Bike Rides by Day of Week"
)

fig.update_layout(
xaxis_title="Day of Week",
yaxis_title="Number of Rides"
)

fig.show()

### Number of Rides by Hour

In [170]:
hourly = (
citibike_df
.groupby("hour", as_index=False)
.agg(number_of_rides=("ride_id","count"))
)

In [171]:
fig = px.line(
hourly,
x="hour",
y="number_of_rides",
markers=True,
title="Hourly Citi Bike Demand"
)

fig.update_layout(
xaxis_title="Hour of Day",
yaxis_title="Number of Rides"
)

fig.show()


### Number of Rides vs Morning / Daytime / Evening

In [172]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'ride_duration_min', 'date', 'month', 'month_name',
       'month_number', 'day_of_week', 'hour', 'season', 'day_part'],
      dtype='str')

In [173]:
part_of_day = (
citibike_df
.groupby("day_part", as_index=False)
.agg(number_of_rides=("ride_id","count"))
)


In [174]:
order = [
"Morning",
"Daytime",
"Evening",
"Night"
]

In [177]:
part_of_day["day_part"] = pd.Categorical(
part_of_day["day_part"],
categories=order,
ordered=True
)

part_of_day = part_of_day.sort_values("day_part")

In [181]:
fig = px.bar(
part_of_day,
x="day_part",
y="number_of_rides",
text_auto=True,
title="Bike Usage by Part of Day"
)

fig.update_layout(
xaxis_title="Part of Day",
yaxis_title="Number of Rides"
)

fig.show()